In [16]:
from google.colab import files
files.upload()


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"bindupautrajyotibrat","key":"14a5bd9c1ec93a544d80563d3108408c"}'}

In [17]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [18]:
!kaggle competitions download -c ieee-fraud-detection
!unzip ieee-fraud-detection.zip


  0% 0.00/118M [00:00<?, ?B/s]
100% 118M/118M [00:00<00:00, 1.58GB/s]
Archive:  ieee-fraud-detection.zip
  inflating: sample_submission.csv   
  inflating: test_identity.csv       
  inflating: test_transaction.csv    
replace train_identity.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: train_identity.csv      
replace train_transaction.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: train_transaction.csv   


In [19]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping


In [20]:
train_transaction = pd.read_csv('train_transaction.csv')
train_identity = pd.read_csv('train_identity.csv')

data = train_transaction.merge(train_identity, how='left', on='TransactionID')


In [21]:
categorical_cols = X.select_dtypes(include=['object']).columns
numeric_cols = X.select_dtypes(exclude=['object']).columns

for col in categorical_cols:
    X[col] = X[col].fillna('missing')
    X[col] = X[col].astype('category').cat.codes

for col in numeric_cols:
    X[col] = X[col].fillna(X[col].median())


In [22]:
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])


In [23]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [24]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))


In [25]:
model = Sequential([
    Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.4),

    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(1, activation='sigmoid')
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [26]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['AUC']
)


In [27]:
early_stop = EarlyStopping(
    monitor='val_auc',
    patience=5,
    mode='max',
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=2048,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 17s 822ms/step - AUC: 0.6383 - loss: 0.8297 - val_AUC: 0.7739 - val_loss: 0.7089
Epoch 2/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - AUC: 0.8204 - loss: 0.5685 - val_AUC: 0.7949 - val_loss: 0.5317
Epoch 3/30


/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_auc` which is not available. Available metrics are: AUC,loss,val_AUC,val_loss
  current = self.get_monitor_value(logs)


12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - AUC: 0.8454 - loss: 0.5152 - val_AUC: 0.8049 - val_loss: 0.4430
Epoch 4/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - AUC: 0.8770 - loss: 0.4679 - val_AUC: 0.8123 - val_loss: 0.4260
Epoch 5/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - AUC: 0.8899 - loss: 0.4389 - val_AUC: 0.8223 - val_loss: 0.3828
Epoch 6/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - AUC: 0.8985 - loss: 0.4146 - val_AUC: 0.8086 - val_loss: 0.3515
Epoch 7/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - AUC: 0.9081 - loss: 0.4014 - val_AUC: 0.8347 - val_loss: 0.3130
Epoch 8/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - AUC: 0.9071 - loss: 0.3968 - val_AUC: 0.8297 - val_loss: 0.2904
Epoch 9/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - AUC: 0.9238 - loss: 0.3692 - val_AUC: 0.8370 - val_loss: 0.2741
Epoch 10/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - AUC: 0.9254 - loss: 0.3647 - val_AUC: 0.8444 - val_loss: 0.2325
Epoch 11/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - AUC: 0.9358 - loss: 

In [28]:
val_preds = model.predict(X_val).ravel()
print("Validation AUC:", roc_auc_score(y_val, val_preds))


182/182 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Validation AUC: 0.8656100989068194
